# Data preprocessing

In [ ]:
%pip install -q -r rag/requirements.txt
%pip uninstall -y torchaudio

In [ ]:
import subprocess
import sys
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO_ROOT = Path.cwd()
RAG_ROOT = REPO_ROOT / 'rag'
DATA_ROOT = REPO_ROOT / 'data'
TARGET = 'rewritten'
RETRIEVER = 'bm25'
TOP_K = 8
DENSE_MODEL = 'BAAI/bge-base-en-v1.5'
MODEL_PATH = 'meta-llama/Llama-3.1-8B-Instruct'
BEGIN_INDEX = 0
END_INDEX = 20600
CONTEXT_LENGTH = 70000
MAX_NEW_TOKENS = 64
MAX_NUM_SEQS = 32
GPU_MEMORY_UTILIZATION = 0.70

DATA_ROOT.mkdir(exist_ok=True)
for name in ('news.tsv', 'personalized_test.tsv'):
    if not (DATA_ROOT / name).exists():
        hf_hub_download('THEATLAS/PENS', name, repo_type='dataset', local_dir=DATA_ROOT)
assert TARGET in {'original', 'rewritten'}
assert RETRIEVER in {'bm25', 'dense', 'random'}

In [ ]:
preprocessed = {}
for target in ('original', 'rewritten'):
    path = DATA_ROOT / f'rank_merge_{target}.json'
    subprocess.run([
        sys.executable, 'data/preprocess.py',
        '--news_file', str(DATA_ROOT / 'news.tsv'),
        '--test_file', str(DATA_ROOT / 'personalized_test.tsv'),
        '--output_file', str(path),
        '--target', target,
    ], cwd=RAG_ROOT, check=True)
    preprocessed[target] = path

retrieval_file = DATA_ROOT / TARGET / f'{RETRIEVER}_{TOP_K}' / f'point_base_{RETRIEVER}.json'
subprocess.run([
    sys.executable, 'ranking.py',
    '--input_path', str(preprocessed[TARGET]),
    '--output_path', str(retrieval_file),
    '--retriever', RETRIEVER,
    '--topk', str(TOP_K),
    '--begin_idx', str(BEGIN_INDEX),
    '--end_idx', str(END_INDEX),
    '--dense_model', DENSE_MODEL,
    '--device', 'cuda',
], cwd=RAG_ROOT, check=True)
print(retrieval_file)

# Generation

In [ ]:
model_name = MODEL_PATH.split('/')[-1]
output_dir = REPO_ROOT / 'outputs' / model_name / TARGET / f'{RETRIEVER}_{TOP_K}'
subprocess.run([
    sys.executable, 'generation/test.py',
    '--input_file', str(retrieval_file),
    '--output_dir', str(output_dir),
    '--model_path', MODEL_PATH,
    '--begin_idx', str(BEGIN_INDEX),
    '--end_idx', str(END_INDEX),
    '--cutoff_len', str(CONTEXT_LENGTH),
    '--max_new_tokens', str(MAX_NEW_TOKENS),
    '--gpu_memory_utilization', str(GPU_MEMORY_UTILIZATION),
    '--max_num_seqs', str(MAX_NUM_SEQS),
], cwd=RAG_ROOT, check=True)
print(output_dir)